<div align="center">

# Patra Toolkit: Model Cards & Datasheets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-to-Insight-Center/patra-toolkit/blob/main/examples/notebooks/ModelCardAndDatasheetDemo.ipynb)

</div>

[Patra](https://github.com/Data-to-Insight-Center/patra-knowledge-base) documents AI/ML models and the datasets that trained them as structured, machine-actionable metadata -- **Model Cards** and **Datasheets** -- instead of a README that goes stale. This notebook walks through the full lifecycle of both using the `patra-toolkit` Python client.

**By the end of this notebook, you'll know how to:**
- Build a Model Card and a Datasheet, and submit either to a Patra server
- Discover and retrieve real records from a running Patra server
- *(Optional, advanced)* Stream a live inference run to Patra through CKN

**Prerequisites:** none beyond Python -- the toolkit installs everything else it needs, and this notebook targets a public demo server so you can run it end-to-end without an account.

### Contents
1. [Setup](#1-setup)
2. [Connect to a Patra server](#2-connect)
3. [Model Cards and Datasheets](#3-cards-and-sheets)
4. [Submit to Patra](#4-submit)
5. [Discover Model Cards & Datasheets](#5-discover)
6. [Optional, advanced: stream a live experiment to CKN](#6-experiment)
7. [Next steps](#7-next-steps)

<a id="1-setup"></a>
## 1. Setup

Install the toolkit, then import the two pieces we'll use first: `ModelCard` for documenting a model, and `AIModel` for its performance and framework details. (`Datasheet` too, since we build one right after.)

In [ ]:
!pip install -q patra-toolkit

In [ ]:
from patra_toolkit import ModelCard, AIModel, Datasheet

<a id="2-connect"></a>
## 2. Connect to a Patra server

Every Model Card and Datasheet is submitted to a **Patra server** -- set its URL once and reuse it for the rest of the notebook. This demo points at a public server, so anonymous submission works out of the box.

Some Patra deployments (e.g. TAPIS-hosted pods) require a JWT for write access. If yours does, leave `tapis_token = None` for now -- [Section 4](#4-submit) covers getting a real one via the client's `authenticate()` method.

In [ ]:
patra_server_url = "https://patrabackenddemo.pods.icicleai.tapis.io/"
tapis_token = None  # set via authenticate() in Section 4 if your server requires it

<a id="3-cards-and-sheets"></a>
## 3. Model Cards and Datasheets

A `ModelCard` documents a model -- what it is, who made it, and how it performs -- built field-by-field; only `name` is required, everything else (including an attached `AIModel` for framework, ownership, and metrics) is optional. A `Datasheet` documents the dataset a model was trained on the same way, with `add_*()` convenience methods (`add_title`, `add_creator`, `add_description`, ...) for its DataCite-style fields. Link the two by pointing a model card's `training_datasheet_uuid` at a submitted datasheet's `uuid`.

The full field reference is in the [README](https://github.com/Data-to-Insight-Center/patra-toolkit#building-a-patra-model-card) and [schema_description.md](https://github.com/Data-to-Insight-Center/patra-toolkit/blob/main/docs/source/schema_description.md) -- [Section 5](#5-discover) below shows what a real submitted record looks like once you look one up.

<a id="4-submit"></a>
## 4. Submit to Patra

Call `validate()` on a `ModelCard` or `Datasheet` to check it against the schema, then `submit(patra_server_url=..., token=...)` -- it re-validates, sends the record to the server, and sets `.uuid` to the id the server assigns.

`submit()` raises `PatraSubmissionError` on a validation or network failure, and `PatraModelExistsError` / `PatraDatasheetExistsError` if an equivalent record already exists on the server -- a Model Card is a duplicate if its `name`, `version`, `author`, and `short_description` all match; a Datasheet, if its title and creator do. Bump `version` to submit a genuinely new one.

Pass `token=<tapis_token>` if your server requires TAPIS authentication (see [Section 2](#2-connect)); get one by calling `.authenticate(username=..., password=...)` on any `ModelCard` instance -- it doesn't touch the instance's data, so any one will do. Either object can also be archived to disk first with `mc.save("model_card.json")` / `ds.save("datasheet.json")`.

There's nothing to run in this section -- the rest of this notebook works against records already on the public demo server, and [Section 5](#5-discover) shows exactly what a submitted record looks like once you look one up.

<a id="5-discover"></a>
## 5. Discover Model Cards & Datasheets

`list_*` returns lightweight summaries, searchable with `q` (a substring match against name/title, author, and short description) and pageable with `skip`/`limit` (server max: 100 per page). `get_*` fetches one full record by `uuid` -- for a Model Card, that includes the nested `ai_model` detail. Both accept `token=` to include private records.

The cells below just pull whatever's already on the public demo server, so they run standalone -- if a list comes back empty, submit a record first (see [Section 4](#4-submit)) and re-run.

### Model Cards

In [ ]:
import pandas as pd

model_cards = ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(model_cards)

Narrow the search with `q`, using a name pulled straight from the results above:

In [ ]:
ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, q=model_cards[0]["name"])

Fetch that record's full detail, including its `ai_model`:

In [ ]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=model_cards[0]["uuid"], token=tapis_token)

### Datasheets

In [ ]:
datasheets = Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(datasheets)

In [ ]:
Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, q=datasheets[0]["title"])

In [ ]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=datasheets[0]["uuid"], token=tapis_token)

<a id="6-experiment"></a>
## 6. Optional, advanced: stream a live experiment to CKN

Everything so far submits *static* metadata. `run_experiment()` goes a step further: it downloads a real pretrained model and a batch of images, runs inference, and streams a metric event per image to **CKN** (Kafka), Patra's runtime event-streaming layer -- so results appear in Patra's web app as they're produced.

This section is independent of Sections 1-5 above and needs more than a Python environment, so treat it as optional.

**This section targets a different Patra server than the rest of the notebook.** CKN's Kafka Connect sink connector writes events into one specific database -- the Model Card it references has to live in that same database, which isn't necessarily where the public demo server (`patra_server_url`, set in Section 2) writes to. Check with whoever runs your CKN deployment which Patra server's database its connector actually targets, and use that one below.

#### Prerequisites
1. A reachable CKN Kafka broker address (e.g. `cknbroker.pods.icicleai.tapis.io:443` -- use whatever your broker's *advertised* external listener actually is, which may not match the port you'd otherwise expect).
2. A `user_id` already registered in the `users` table wherever these events land -- `run_experiment()` does not auto-register users, and there's no safe default.
3. The `experiments` extra: `pip install "patra-toolkit[experiments]"` (installs torch, torchvision, pillow, and confluent-kafka).

`run_experiment()` connects over SSL by default (`use_ssl=True`) -- brokers reached through a TLS-terminating proxy (like a Tapis Pod's external port) need this even if the broker's own internal listener config says `PLAINTEXT`, since that setting only describes the broker's side of the connection *after* the proxy's TLS termination. Pass `use_ssl=False` for a broker with no such proxy in front of it.

If your CKN deployment's Kafka Connect sink connector doesn't require the schema-enveloped JSON format (some do, some accept bare JSON), pass `use_schema_envelope=False` to `run_experiment()`.

In [ ]:
!pip install -q "patra-toolkit[experiments]"

### 6.1 Build and submit a Model Card + Datasheet for the inference model

`run_experiment()` (below) looks up a Model Card and Datasheet by uuid, so it needs its own dedicated pair, built here -- one with a real, downloadable model behind `ai_model.location` and `ai_model.inference_labels` set, and the other pointing at a real image source.

`ai_model.location` is set to torchvision's real, publicly downloadable MobileNetV2 weights URL. Reading `weights.url` below doesn't download anything -- `run_experiment()` does the actual download later.

`ckn_patra_server_url` is the Patra server whose database CKN's connector actually writes to -- replace it with the right one for your deployment.

In [ ]:
import torchvision
from patra_toolkit import run_experiment
from patra_toolkit.datasheet import DatasheetAlternateIdentifier

weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1
weights_url = weights.url
imagenet_categories = weights.meta["categories"]
top1_acc = weights.meta["_metrics"]["ImageNet-1K"]["acc@1"] / 100.0

inference_ai_model = AIModel(
    name="MobileNetV2_ImageNet",
    version="1.0",
    description="Torchvision MobileNetV2 CNN pretrained on ImageNet-1k; used for a live inference-streaming demo.",
    owner="Demo Author",
    location=weights_url,
    license="BSD-3-Clause",
    framework="pytorch",
    model_type="cnn",
    test_accuracy=round(top1_acc, 5),
    inference_labels=imagenet_categories,
)

inference_mc = ModelCard(
    name="MobileNetV2_Inference_Demo",
    version="1.0",
    short_description="Real pretrained MobileNetV2 used to demonstrate inference + CKN streaming.",
    full_description=(
        "Downloads a real ImageNet-pretrained MobileNetV2 checkpoint via ai_model.location, runs it "
        "over sample images, and streams per-image inference metrics to CKN."
    ),
    keywords="demo, patra, ckn, inference, mobilenetv2",
    author="Demo Author",
    input_type="Image",
    category="classification",
)
inference_mc.ai_model = inference_ai_model
inference_mc.validate()

inference_ds = Datasheet(publication_year=2026, version="1.0")
inference_ds.add_title("CKN Inference Demo Images")
inference_ds.add_creator("Demo Author")
inference_ds.alternate_identifiers.append(
    DatasheetAlternateIdentifier(alternate_identifier="https://picsum.photos", alternate_identifier_type="URL")
)
inference_ds.add_description(
    "Images fetched from Lorem Picsum via https://picsum.photos/id/{n}/224/224 for a live "
    "inference-streaming demo. These are arbitrary real-world photographs with no ImageNet ground-truth labels.",
    "TechnicalInfo",
)
inference_ds.validate()

In [ ]:
ckn_patra_server_url = "https://patrabackend.pods.icicleai.tapis.io/"

inference_mc_result = inference_mc.submit(patra_server_url=ckn_patra_server_url, token=tapis_token)
inference_ds_result = inference_ds.submit(patra_server_url=ckn_patra_server_url, token=tapis_token)
print("Model Card uuid:", inference_mc.uuid)
print("Datasheet uuid:", inference_ds.uuid)

### 6.2 Run the experiment

`run_experiment()` does the rest on its own: downloads the Model Card and Datasheet by uuid, downloads the model weights and sample images they reference, runs inference, and streams a CKN event per image as it's processed.

`categories` is passed explicitly (reusing `imagenet_categories` from 6.1) rather than relying on the fetched Model Card -- `GET /modelcard/{uuid}` doesn't echo back `ai_model.inference_labels`, so it can't be re-derived after the round trip.

In [ ]:
result = run_experiment(
    model_card_uuid=inference_mc.uuid,
    datasheet_uuid=inference_ds.uuid,
    patra_server_url=ckn_patra_server_url,
    ckn_broker_url="cknbroker.pods.icicleai.tapis.io:443",  # use your CKN broker's advertised external address
    user_id="demo_user",  # replace with your own registered user_id
    token=tapis_token,
    categories=imagenet_categories,  # GET /modelcard doesn't echo back ai_model.inference_labels, so pass it directly
)
result

### 6.3 View results in the Patra web app

1. **Backend**: `ENABLE_DOMAIN_EXPERIMENTS` needs to be enabled on `ckn_patra_server_url` (the server this experiment was streamed against, not necessarily `patra_server_url` from Section 2).
2. **Web app**: set `VITE_SUPPORTS_DOMAIN_EXPERIMENTS=true` in `patra-frontend/app/.env` (it defaults to `false`), then run `npm run dev` from `patra-frontend/app/`.
3. Open the app and click **Digital Agriculture** under Experiments in the sidebar, then select your `user_id` to see this run's summary and per-image results.

You can also check the same data the web app reads directly via the REST API -- `result['results_url']` above is exactly that endpoint:
```bash
curl -s "$(python3 -c "print(result['results_url'])")"
```

**If nothing shows up**, check your CKN deployment's Kafka Connect logs for errors around the time you ran this -- the sink connector silently drops malformed or unresolvable records (e.g. an unregistered `user_id` or `model_id`, or a schema-envelope mismatch -- see `use_schema_envelope` above) rather than raising anything visible here.

<a id="7-next-steps"></a>
## 7. Next steps

- **Fairness & explainability**: `mc.populate_bias(...)` (via [fairlearn](https://fairlearn.org/)) and `mc.populate_xai(...)` (via [SHAP](https://shap.readthedocs.io/)) can auto-populate bias and feature-importance metrics from a trained model -- see the [README](https://github.com/Data-to-Insight-Center/patra-toolkit#run-fairness-and-explainability-scanners).
- **Schema reference**: every field on `ModelCard` and `AIModel` is documented in [schema_description.md](https://github.com/Data-to-Insight-Center/patra-toolkit/blob/main/docs/source/schema_description.md).
- **More examples**: framework-specific walkthroughs (PyTorch, TensorFlow, scikit-learn, Hugging Face) live alongside this notebook in [examples/notebooks/](https://github.com/Data-to-Insight-Center/patra-toolkit/tree/main/examples/notebooks).
- **Browse submitted records**: Patra's web app, in this workspace at `patra-frontend/`, lets you search, view, and edit Model Cards and Datasheets from a browser.